# Load Southampton's LSOAs into the `area` table

Reads the ONS lookup "LSOA (2021) to Electoral Ward to LAD Best Fit Lookup in EW" and fills the `area` table with one row per LSOA in Southampton.

Notes
- The `area` table sits in the same `business_ecosystem.db` as `business_type`.
- `lsoa_code` is the official ONS code (for example `E01017224`). Every outside dataset (Census, business counts, postcodes) joins on it, so it is unique and never changes. `id` is just the internal key that other tables will point at.
- LSOA names look like `Southampton 024C`, which is not very friendly. The ward name (Peartree, Portswood, ...) is better for labels on charts and maps.
- Ward boundaries change every year or two, so treat ward as descriptive only. Don't join other data on it.
- To add fringe areas later (Eastleigh, Totton), add the council name to `TARGET_LADS` and re-run. Existing rows keep their `id`.
- Safe to re-run: rows are matched on `lsoa_code`, so `id` values never change.

## 1. Settings

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

CSV_PATH = Path("raw/official/LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.csv")
DB_PATH = Path("processed/database.db")   # same database as the business_type notebook
TARGET_LADS = ["Southampton"]             # local authority names to keep

## 2. Read the CSV and find the columns

In [2]:
# utf-8-sig strips the invisible byte-order mark at the start of this file,
# which would otherwise corrupt the first column name
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
raw.columns = [c.strip() for c in raw.columns]


def pick(pattern):
    # Column names contain the edition year (WD24CD, LAD24NM). Match on the pattern
    # so a newer edition of the file (WD25CD, ...) still works.
    hits = [c for c in raw.columns if re.fullmatch(pattern, c)]
    assert len(hits) == 1, f"Expected exactly one column matching {pattern}, found {hits}"
    return hits[0]


col = {
    "lsoa_code": pick(r"LSOA\d{2}CD"),
    "name": pick(r"LSOA\d{2}NM"),
    "ward_code": pick(r"WD\d{2}CD"),
    "ward_name": pick(r"WD\d{2}NM"),
    "lad_code": pick(r"LAD\d{2}CD"),
    "lad_name": pick(r"LAD\d{2}NM"),
}
print(f"{len(raw):,} rows read (all of England and Wales)")
print(col)

35,672 rows read (all of England and Wales)
{'lsoa_code': 'LSOA21CD', 'name': 'LSOA21NM', 'ward_code': 'WD24CD', 'ward_name': 'WD24NM', 'lad_code': 'LAD24CD', 'lad_name': 'LAD24NM'}


## 3. Keep the target council(s) and validate

In [3]:
area = raw[list(col.values())].rename(columns={v: k for k, v in col.items()})
area = area.apply(lambda s: s.str.strip())
area = area[area["lad_name"].isin(TARGET_LADS)].copy()

not_found = set(TARGET_LADS) - set(area["lad_name"])
assert not not_found, f"No rows found for {not_found}. Check the spelling against the lad_name column."

bad_codes = area[~area["lsoa_code"].str.fullmatch(r"E01\d{6}")]
assert bad_codes.empty, f"Codes that don't look like English LSOA codes:\n{bad_codes}"
assert area["lsoa_code"].is_unique, "Duplicate LSOA codes in the selection"

blanks = (area == "").sum()
assert blanks.sum() == 0, f"Blank values found:\n{blanks[blanks > 0]}"

assert 50 <= len(area) <= 400, f"{len(area)} areas looks wrong for one or two councils, check the filter"

print(f"{len(area)} areas selected")
print(area.groupby("lad_name").size())
area.head()

152 areas selected
lad_name
Southampton    152
dtype: int64


,lsoa_code,name,ward_code,ward_name,lad_code,lad_name
5200,E01017224,Southampton 024C,E05015499,Peartree,E06000045,Southampton
5202,E01017220,Southampton 028C,E05015499,Peartree,E06000045,Southampton
5204,E01017223,Southampton 024B,E05015499,Peartree,E06000045,Southampton
5206,E01017225,Southampton 024D,E05015499,Peartree,E06000045,Southampton
5209,E01017235,Southampton 013E,E05015500,Portswood,E06000045,Southampton


## 4. Create the table

In [4]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS area (
    id         INTEGER PRIMARY KEY,
    lsoa_code  TEXT NOT NULL UNIQUE CHECK (length(lsoa_code) = 9),
    name       TEXT NOT NULL,
    ward_code  TEXT,
    ward_name  TEXT,
    lad_code   TEXT NOT NULL,
    lad_name   TEXT NOT NULL
);
"""

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON;")   # SQLite ignores foreign keys unless this is set on every connection
con.execute("PRAGMA journal_mode = WAL;")  # lets teammates read while you write
con.executescript(SCHEMA)

## 5. Load the rows

In [5]:
columns = ["lsoa_code", "name", "ward_code", "ward_name", "lad_code", "lad_name"]
rows = list(area[columns].itertuples(index=False, name=None))

with con:   # one transaction: all rows are saved, or none are
    con.executemany(
        """
        INSERT INTO area (lsoa_code, name, ward_code, ward_name, lad_code, lad_name)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(lsoa_code) DO UPDATE SET
            name      = excluded.name,
            ward_code = excluded.ward_code,
            ward_name = excluded.ward_name,
            lad_code  = excluded.lad_code,
            lad_name  = excluded.lad_name
        """,
        rows,
    )

print(f"Loaded {len(rows)} rows into {DB_PATH}")

Loaded 152 rows into processed/database.db


## 6. Check the result

In [6]:
loaded = {r[0] for r in con.execute("SELECT lsoa_code FROM area")}
assert set(area["lsoa_code"]) <= loaded, "Some selected areas did not make it into the table"

pd.read_sql("SELECT lad_name, COUNT(*) AS areas FROM area GROUP BY lad_name", con)

,lad_name,areas
0,Southampton,152


Areas per ward. Each ward should have a handful, and no ward should be missing:

In [7]:
pd.read_sql(
    """
    SELECT ward_name, ward_code, COUNT(*) AS areas
    FROM area
    GROUP BY ward_name, ward_code
    ORDER BY ward_name
    """,
    con,
)

,ward_name,ward_code,areas
0,Banister & Polygon,E05015490,9
1,Bargate,E05015491,9
2,Bassett,E05015492,9
3,Bevois,E05015493,9
4,Bitterne Park,E05015494,10
5,Coxford,E05015495,9
6,Freemantle,E05015496,7
7,Harefield,E05015497,8
8,Millbrook,E05015498,9
9,Peartree,E05015499,9


In [8]:
con.close()